# Introduction

This project implements a CNN (Convolutional Neural Network) for the classification of long phrases written by two authors, Kant and Freud (Spanish translation).

Although the classification task was relatively simple, the main objective of this project was to adapt and apply the code presented in the manual to solve similar tasks using locally stored data.

The CNN was implemented using TensorFlow 2.20.0, with modifications to adapt the original implementation to the specific task, dataset, and deployment requirements.

The original code was developed by Ganegedara (2022) in the book *Natural Language Processing with TensorFlow 2*.

GitHub repository: https://github.com/thushv89/packt_nlp_tensorflow_2

In [ ]:
# Imports
%matplotlib inline

import os
import json
import random
import re

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
import tensorflow.keras.backend as K
import tensorflow.keras.layers as layers
import tensorflow.keras.regularizers as regularizers
from tensorflow.keras.models import Model


# Reproducibility
seed = 54321

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

%env TF_FORCE_GPU_ALLOW_GROWTH=true

# Data wrangling

## Text cleaning and data splitting

In [ ]:
# Read data
def read_data(data_dir):
    pages_list = []    
    filenames = []
    print("Reading files...")

    i = 0
    for root, dirs, files in os.walk(data_dir):
        for fi, f in enumerate(files):
            if 'readme' in f.lower():
                continue
            
            i += 1
            #print("." * i, f, end='\r')

            file_path = os.path.join(root, f)
            with open(file_path, encoding='utf-8') as text_file:
                lines = [line.strip() for line in text_file]
                page = ' '.join(lines)
                pages_list.append(page)
                filenames.append(file_path)

    print(f"\nDetected {len(pages_list)} text files.")
    return pages_list, filenames

# Path where the two categories folder are
folder_pages = r'C:\Users\juanm\Jupyter nootebooks\Curso IA\cnn_authorclass_serving\data'


data, filenames = read_data(folder_pages)

# FIles counts
print(f"{sum(len(page.split()) for page in data)} total words.")
print("Example (start):", data[0][:50])
print("Example (end):", data[-1][-50:])

In [ ]:
# Paths
base_dir = folder_pages
kant_dir = os.path.join(base_dir, "Kant") # Folder name = Kant
freud_dir = os.path.join(base_dir, "Freud") # Folder name = Freud

# Read text files and assign labels
pages = []
labels = []

def read_txts(ruta, etiqueta):
    for filename in sorted(os.listdir(ruta)):
        if filename.lower().endswith(".txt"):
            ruta_completa = os.path.join(ruta, filename)
            with open(ruta_completa, encoding="utf-8") as f:
                text = f.read().strip()
                pages.append(text)
                labels.append(etiqueta)

# Load texts from each author
read_txts(kant_dir, "Kant") # Label = Kant
read_txts(freud_dir, "Freud") # Label = Freud

# Random train-test split
train_pages, test_pages, train_categories, test_categories = train_test_split(
    pages, labels, test_size=0.2, random_state=42
)

# Build dataframes
train_df = pd.DataFrame({'text': train_pages, 'category': train_categories})
test_df = pd.DataFrame({'text': test_pages, 'category': test_categories})

# Display first examples
train_df.head(10)

In [ ]:
# Words and phrases to remove during preprocessing

words_to_replace = {
    "capítulo",
    "editorial",
    "introducción",
    "Página",
    "freud",
    "kant",
    "www.lectulandia.com",
    "página",
    "prolegomenos"
}

phrases_to_replace = [
    "obras completas",
    "sigmund freud",
    "immanuel kant",
]


def clean_text(text, words=None, frases=None):

    if words is None:
        words = words_to_replace

    if frases is None:
        frases = phrases_to_replace

    # Convert text to lowercase
    text = text.lower()

    # Remove complete phrases
    for frase in frases:
        text = text.replace(frase.lower(), "")

    # Replace special characters
    text = re.sub(r"[«»“”‘’—–…]", " ", text)

    # Normalize common abbreviations
    text = re.sub(r'\bq\b', 'que', text)

    # Remove individual words
    for word in words:
        text = re.sub(rf'\b{re.escape(word)}\b', '', text)

    # Remove numbers and dates
    text = re.sub(r'\b\d+(?:[\.,]\d+)?\b', '', text)

    # Keep only letters and spaces
    text = re.sub(r'[^a-záéíóúüñ\s]', '', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    # Remove first and last 10 words
    words_list = text.split()

    # Remove 
    if len(words_list) > 20:
        words_list = words_list[10:-10]

    return " ".join(words_list)

In [ ]:
# Apply text cleaning function
train_df['text'] = train_df['text'].apply(lambda x: clean_text(x, words=words_to_replace, frases=phrases_to_replace))
test_df['text'] = test_df['text'].apply(lambda x: clean_text(x, words=words_to_replace, frases=phrases_to_replace))

In [ ]:
# Shuffle data
train_df = train_df.sample(frac=1.0, random_state=seed) # frac=1.0 means using 100% of the data
test_df = test_df.sample(frac=1.0, random_state=seed)

In [ ]:
# Explicit label mapping
labels_map = {
    "Freud": 0,
    "Kant": 1
}

n_classes = len(labels_map)

print(f"Label -> ID mapping: {labels_map}")

# Convert string labels into numeric labels
train_df["category"] = train_df["category"].map(labels_map)
test_df["category"] = test_df["category"].map(labels_map)

train_df.head(n=10)

In [ ]:
# Split training data into training and validation sets
train_df, valid_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=seed
)

print(f"Train size: {train_df.shape}")
print(f"Validation size: {valid_df.shape}")

train_df.head()

## Tokenization

In [ ]:
# Initialize tokenizer and fit only on training data
tokenizer = Tokenizer()

tokenizer.fit_on_texts(
    train_df["text"].tolist()
)

# Vocabulary size
n_vocab = len(tokenizer.word_index) + 1

print(f"Vocabulary size: {n_vocab}")

Looking for the longest sentence

In [ ]:
# Compute token length distribution and selected percentiles
train_df["text"].str.split(" ").str.len().describe(percentiles=[0.01, 0.5, 0.99])

Padding for setences

In [ ]:
# Convert text into sequences of token IDs

train_sequences = tokenizer.texts_to_sequences(
    train_df["text"].tolist()
)

valid_sequences = tokenizer.texts_to_sequences(
    valid_df["text"].tolist()
)

test_sequences = tokenizer.texts_to_sequences(
    test_df["text"].tolist()
)

train_labels = train_df["category"].values
valid_labels = valid_df["category"].values
test_labels = test_df["category"].values


# Maximum sequence length selected from the 99th percentile
max_seq_length = 599


# Pad or truncate sequences
preprocessed_train_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    train_sequences,
    maxlen=max_seq_length,
    padding="post",
    truncating="post"
)

preprocessed_valid_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    valid_sequences,
    maxlen=max_seq_length,
    padding="post",
    truncating="post"
)

preprocessed_test_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    test_sequences,
    maxlen=max_seq_length,
    padding="post",
    truncating="post"
)

In [ ]:
K.clear_session()

# Input layer using word IDs
word_id_inputs = layers.Input(
    shape=(max_seq_length,),
    dtype="int32"
)

# Embedding layer
embedding_out = layers.Embedding(
    input_dim=n_vocab,
    output_dim=64
)(word_id_inputs)


# Convolutional layers
conv1_1 = layers.Conv1D(
    100,
    kernel_size=3,
    strides=1,
    padding="same",
    activation="relu"
)(embedding_out)

conv1_2 = layers.Conv1D(
    100,
    kernel_size=4,
    strides=1,
    padding="same",
    activation="relu"
)(embedding_out)

conv1_3 = layers.Conv1D(
    100,
    kernel_size=5,
    strides=1,
    padding="same",
    activation="relu"
)(embedding_out)


# Concatenate convolution outputs
conv_out = layers.Concatenate(axis=-1)(
    [conv1_1, conv1_2, conv1_3]
)


# Max pooling
pool_over_time_out = layers.MaxPool1D(
    pool_size=max_seq_length,
    padding="valid"
)(conv_out)


# Flatten
flatten_out = layers.Flatten()(pool_over_time_out)


# Output layer
out = layers.Dense(
    n_classes,
    activation="softmax",
    kernel_regularizer=regularizers.l2(0.001)
)(flatten_out)


# Build model
cnn_model = Model(
    inputs=word_id_inputs,
    outputs=out
)


# Compile model
cnn_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

cnn_model.summary()

# Training

As stated previously, the training process was relatively simple because the main objective of this project was to adapt the original code from the manual to work with locally stored data. Therefore, only five epochs were used.

In [ ]:
# Learning rate reduction callback
lr_reduce_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.1,
    patience=3,
    verbose=1,
    mode="auto",
    min_delta=0.0001,
    min_lr=0.000001
)


# Train model
history_cnn_model = cnn_model.fit(
    preprocessed_train_sequences,
    train_labels,
    validation_data=(
        preprocessed_valid_sequences,
        valid_labels
    ),
    batch_size=128,
    epochs=5,
    callbacks=[lr_reduce_callback]
)

In [ ]:
# Evaluate model accuracy on the test set
test_results = cnn_model.evaluate(
    preprocessed_test_sequences,
    test_labels,
    return_dict=True
)

test_results

In [ ]:
# Save trained model
cnn_model.save("cnn_model.keras")

In [ ]:
# Save tokenizer for inference

tokenizer_json = tokenizer.to_json()

with open("tokenizer.json", "w", encoding="utf-8") as f:
    f.write(tokenizer_json)

print("Tokenizer saved.")

In [ ]:
# Save inference configuration

inference_config = {
    "max_seq_length": max_seq_length,
    "label_map": {
        "0": "Freud",
        "1": "Kant"
    },
    "padding": "post",
    "truncating": "post"
}

with open("inference_config.json", "w", encoding="utf-8") as f:
    json.dump(
        inference_config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Inference configuration saved.")

## Export model

In [ ]:
# Export model for TensorFlow Serving
export_path = r'C:\Users\juanm\Jupyter nootebooks\Curso IA\cnn_authorclass_serving'

model_name = "kant_freud_model"
model_version = 1

export_path = os.path.join(
    model_name,
    str(model_version)
)

cnn_model.export(export_path)

print(f"SavedModel exported to: {export_path}")

## Test save model

In [ ]:
# Test the exported SavedModel

loaded_saved_model = tf.saved_model.load(export_path)

print("Available signatures:")
print(list(loaded_saved_model.signatures.keys()))

## Test inferencia

In [ ]:
# Test inference using the exported SavedModel

serving_fn = loaded_saved_model.signatures["serve"]

sample_input = tf.constant(
    preprocessed_test_sequences[:2],
    dtype=tf.int32
)

sample_output = serving_fn(sample_input)

sample_output

In [ ]:
keras_predictions = cnn_model.predict(
    preprocessed_test_sequences[:2]
)

In [ ]:
savedmodel_predictions = sample_output[
    list(sample_output.keys())[0]
].numpy()

print("Keras predictions:")
print(keras_predictions)

print("\nSavedModel predictions:")
print(savedmodel_predictions)

In [ ]:
np.testing.assert_allclose(
    keras_predictions,
    savedmodel_predictions,
    rtol=1e-5,
    atol=1e-5
)

print("Keras and SavedModel predictions match.")

# Csv and xslx file with clasifications

In [ ]:
# Load model if the kernel was restarted
cnn_model = tf.keras.models.load_model("cnn_model.keras")

In [ ]:
# Generate predictions on the test set
test_predictions = cnn_model.predict(preprocessed_test_sequences)

## Correct and incorrect predictions

In [ ]:
# Get predicted classes from softmax outputs
predicted_labels = np.argmax(test_predictions, axis=1)

# argmax returns the index of the highest value
# in an array or tensor

# Boolean masks for correct and incorrect predictions
correct_mask = (predicted_labels == test_labels)
incorrect_mask = (predicted_labels != test_labels)

# Count predictions
num_correct = np.sum(correct_mask)
num_incorrect = np.sum(incorrect_mask)

print(f"Correctly classified examples: {num_correct}")
print(f"Incorrectly classified examples: {num_incorrect}")

In [ ]:
# Check label encoding
unique_labels = np.unique(test_labels)
print(unique_labels)

In [ ]:
# Create a dataframe containing classification results and prediction probabilities

def df_classification_results_with_probs(test_texts, test_labels, test_predictions, label_map, n_samples=10):
    pred_labels = np.argmax(test_predictions, axis=1)
    correct_mask = (pred_labels == test_labels)

    rows = []
    n_classes = len(label_map)
    for text, true_label, pred_label, probs, correct in zip(test_texts, test_labels, pred_labels, test_predictions, correct_mask):
        # Format probabilities with 3 decimals, example: {'Freud': 0.923, 'Kant': 0.077}
        prob_dict = {label_map[i]: f"{probs[i]:.3f}" for i in range(n_classes)}
        rows.append({
            "Text": text,
            "True Label": label_map[true_label],
            "Predicted Label": label_map[pred_label],
            "Correct": correct,
            "Probabilities": prob_dict
        })

    df = pd.DataFrame(rows)

    print("Correct examples:")
    display(df[df["Correct"]].head(n_samples))
    
    print("Incorrect examples:")
    display(df[~df["Correct"]].head(n_samples))

    return df # return dataframe for further analysis and saving

In [ ]:
# Display classification results and prediction probabilities

results_df = df_classification_results_with_probs(
    test_texts=test_df["text"].tolist(),
    test_labels=test_df["category"].to_numpy(),
    test_predictions=test_predictions,
    label_map={0: "Freud", 1: "Kant"},
)

# save the results in csv format
results_df.to_csv(r'C:\Users\juanm\Jupyter nootebooks\Curso IA\cnn_authorclass_serving\df_clasificaciones.csv', sep = ';', encoding= 'latin-1',index=False)
results_df.to_excel(r'C:\Users\juanm\Jupyter nootebooks\Curso IA\cnn_authorclass_serving\df_clasificaciones.xlsx',index=False)

## Extra tests

In [ ]:
print("Test samples:", len(test_labels))
print("Predictions:", len(predicted_labels))

print("Unique true labels:", np.unique(test_labels, return_counts=True))
print("Unique predicted labels:", np.unique(predicted_labels, return_counts=True))

In [ ]:
train_texts = set(train_df["text"])
test_texts = set(test_df["text"])

overlap = train_texts.intersection(test_texts)

print("Train/Test exact duplicates:", len(overlap))

In [ ]:
text_kant = """
La razón busca establecer principios universales mediante los cuales
el conocimiento pueda distinguirse de la mera experiencia sensible.
La libertad no consiste simplemente en actuar según los deseos,
sino en someter la voluntad a una ley que la razón pueda reconocer
como universal.
"""

In [ ]:
text_freud = """
El deseo reprimido puede retornar de manera indirecta y manifestarse
en los sueños, los síntomas y otros actos que escapan al control
consciente del sujeto. El conflicto entre la vida psíquica consciente
y aquello que ha sido reprimido puede producir diversas formaciones
del inconsciente.
"""

In [ ]:
text_mixed = """
La razón pretende gobernar la conducta mediante principios universales,
pero detrás de esa aparente autonomía de la voluntad pueden encontrarse
deseos inconscientes que determinan silenciosamente nuestras decisiones.
La conciencia cree actuar libremente, aunque una parte de la vida
psíquica permanece fuera de su conocimiento.
"""

In [ ]:
text_ambiguous = """
El sujeto busca comprender los principios que orientan su conducta,
pero no siempre puede conocer las causas que determinan sus acciones.
Aquello que creemos elegir conscientemente puede estar condicionado
por procesos que no reconocemos de manera inmediata.
"""

In [ ]:
text_ridiculo = """
Me comrpo una garompa y todo me importa muy poco. El otro dia comi una uva y no me gusto, me parece
que me dio nausaeas. Pero bueno me gusta el jamon. La Wikipedia en español es la edición en español o castellano de Wikipedia. 
Al igual que las versiones existentes de Wikipedia en otros idiomas, 
es una enciclopedia de contenido libre, publicada en Internet bajo las licencias libres CC BY-SA 4.0 y GFDL. 
En la actualidad cuenta con 2 138 114 artículos, y es escrita por usuarios voluntarios, es decir, que cualquiera puede editar 
un artículo, corregirlo o ampliarlo. Los servidores son administrados por la Fundación Wikimedia, una organización sin ánimo de lucro cuya 
financiación se basa fundamentalmente en donaciones.
Comenzó el 20 de mayo de 2001, cuatro meses después de lanzarse la edición original en inglés y tras el anuncio de Jimmy Wales de internacionalizar 
el proyecto. Es una de las diez Wikipedias con más artículos de entre todos los idiomas.
"""

In [ ]:
texto_corto_freud = """
El inconsciente esta estructurado como un lenguaje
"""

In [ ]:
texto_corto_kant = """
La razon no aveces no alcanza
"""

In [ ]:
texto_corto_ambiguo = """
No tengo la menor idea que escribir aca
"""

In [ ]:
texto_corto_kant = """
La razón práctica determina la voluntad mediante principios universales
"""

In [ ]:
texto_mezclado = """
La razón determina la voluntad, pero el deseo inconsciente condiciona la conducta
"""

In [ ]:
texto_razon = """
La razón determina la voluntad mediante principios universales.
"""

In [ ]:
texto_deseo = """
El deseo inconsciente determina la conducta mediante procesos psíquicos.
"""

In [ ]:
texto_mezcla_2 = """
La razón determina la voluntad, pero el deseo inconsciente condiciona la conducta.
"""

In [ ]:
texto_freud_razon = """
La razón se convierte en una enemiga que nos priva de tantas posibilidades
de placer. Descubrimos cuánto placer procura escapar a ella por lo menos
temporalmente y entregarse a las seducciones de lo insensato.
"""

In [ ]:
texto_freud_razon_largo = """
La razón se convierte en una enemiga que nos priva de tantas posibilidades
de placer. Descubrimos cuánto placer procura escapar a ella por lo menos
temporalmente y entregarse a las seducciones de lo insensato. El deseo
inconsciente encuentra así caminos para expresarse y busca satisfacción
a través de pensamientos, sueños y actos que no son plenamente conscientes.
"""

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_text(text):
    cleaned = clean_text(text)

    sequence = tokenizer.texts_to_sequences([cleaned])

    padded = pad_sequences(
        sequence,
        maxlen=max_seq_length,
        padding="post",
        truncating="post"
    )

    probabilities = cnn_model.predict(padded, verbose=0)[0]

    print(f"Freud: {probabilities[0]:.4f}")
    print(f"Kant:  {probabilities[1]:.4f}")
    print(f"Predicción: {'Freud' if np.argmax(probabilities) == 0 else 'Kant'}")

In [ ]:
predict_text(texto_freud_razon_largo)